# Test 3 and Test 4 analysis

This notebook combines the completed **F1–F5, 5 ms** Test 3a runs. Test 3b is deliberately excluded from inference because a 5 ms window does not resolve the timestamp modifier it was intended to test.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path("C:/Users/cxm3593/Academic/Workspace/EventSimilarityAnalysis/output/tests")
FREQUENCIES = ["F1", "F2", "F3", "F4", "F5"]
FREQUENCY_COLOURS = {
    "F1": "#d1352b", "F2": "#e07b28", "F3": "#2f9e44",
    "F4": "#1f6fb4", "F5": "#7b3ea1",
}
METRIC_ORDER = ["mmd_rbf03", "mmd_rbf15", "mmd_rbf75", "swd", "chamfer"]
METRIC_LABELS = {
    "mmd_rbf03": "MMD RBF-3", "mmd_rbf15": "MMD RBF-15",
    "mmd_rbf75": "MMD RBF-75", "swd": "SWD", "chamfer": "Chamfer",
}
METRIC_COLOURS = {
    "mmd_rbf03": "#2a78d6", "mmd_rbf15": "#eb6834",
    "mmd_rbf75": "#1baf7a", "swd": "#eda100", "chamfer": "#4a3aa7",
}

TEST3A_RUNS = {
    "F1": ROOT / "optical_chopper_data_f1/test3_modifier/20260825_220306_3a_w5000us",
    "F2": ROOT / "optical_chopper_data_f2/test3_modifier/20260825_222715_3a_w5000us",
    "F3": ROOT / "optical_chopper_data_f3/test3_modifier/20260825_234420_3a_w5000us",
    "F4": ROOT / "optical_chopper_data_f4/test3_modifier/20260826_023134_3a_w5000us",
    "F5": ROOT / "optical_chopper_data_f5/test3_modifier/20260826_071708_3a_w5000us",
}

def read_run(folder):
    folder = Path(folder)
    config = yaml.safe_load((folder / "run_config.yaml").read_text(encoding="utf-8"))
    return {
        "folder": folder,
        "config": config,
        "summary": pd.read_csv(folder / "summary.csv"),
    }

test3a_runs = {frequency: read_run(path) for frequency, path in TEST3A_RUNS.items()}


## Test 3a — controlled modifier sweeps across F1–F5

Every panel retains the metric's raw units. Colour identifies the recording frequency, consistently across all panels. This view asks whether a response is monotonic and whether the same modification produces a stable reading across event rates.


In [2]:
MODIFIER_ORDER = [
    "spatial_offset_x", "spatial_offset_xy", "scaling",
    "subsample", "uniform_noise", "temporal_clump_uniform",
]
MODIFIER_LABELS = {
    "spatial_offset_x": "spatial offset x (px)",
    "spatial_offset_xy": "spatial offset x+y (px)",
    "scaling": "spatial scale",
    "subsample": "subsampling",
    "uniform_noise": "uniform noise",
    "temporal_clump_uniform": "temporal clump (µs)",
}

test3a_summary = []
for frequency, bundle in test3a_runs.items():
    frame = bundle["summary"].copy()
    frame["frequency"] = frequency
    test3a_summary.append(frame)
test3a_summary = pd.concat(test3a_summary, ignore_index=True)

def selectivity_overlay(summary, modifiers=MODIFIER_ORDER, metrics=METRIC_ORDER):
    figure = make_subplots(
        rows=len(modifiers), cols=len(metrics),
        horizontal_spacing=0.035, vertical_spacing=0.055,
        subplot_titles=[
            METRIC_LABELS[metric] if row == 0 else ""
            for row in range(len(modifiers)) for metric in metrics
        ],
    )
    for row, modifier in enumerate(modifiers, start=1):
        for col, metric in enumerate(metrics, start=1):
            for frequency in FREQUENCIES:
                block = summary[(summary.modifier == modifier) &
                                (summary.metric == metric) &
                                (summary.frequency == frequency)].sort_values("magnitude")
                if block.empty:
                    continue
                figure.add_trace(go.Scatter(
                    x=block.magnitude, y=block.mean_distance, mode="lines+markers",
                    name=frequency, legendgroup=frequency,
                    showlegend=(row == 1 and col == 1),
                    line=dict(color=FREQUENCY_COLOURS[frequency], width=1.8),
                    marker=dict(size=4),
                    error_y=dict(type="data", array=block.sd_distance,
                                 thickness=0.6, width=0, visible=True),
                    customdata=np.stack([block.sd_distance, block.n_comparisons], axis=-1),
                    hovertemplate=(frequency + "<br>magnitude %{x:g}<br>mean %{y:.5g}"
                                   + "<br>sd %{customdata[0]:.4g}"
                                   + "<br>n %{customdata[1]:,.0f}<extra></extra>"),
                ), row=row, col=col)
            if col == 1:
                figure.update_yaxes(title_text=MODIFIER_LABELS[modifier], row=row, col=col)
    figure.update_layout(
        title=("Test 3a — metric selectivity across F1–F5"
               "<br><sup>5 ms windows; raw metric units; colour identifies recording</sup>"),
        template="plotly_white", height=225 * len(modifiers), width=1250,
        margin=dict(l=155, r=145, t=105, b=50),
        legend=dict(orientation="v", xanchor="left", x=1.01,
                    yanchor="top", y=1.0, title_text="Recording"),
    )
    return figure

selectivity_overlay(test3a_summary).show()


## Test 3b — not interpreted from the 5 ms run

The directed timestamp-clumping hypothesis concerns structure much shorter than 5 ms. A 5 ms comparison window combines too much surrounding activity to isolate that modifier. The current Test 3b output can validate plotting and data-loading code, but it should not support a scientific conclusion. A future rerun should choose its window from the temporal scale of the modifier itself.


# Test 4 — build one result at a time

The intended structure has three parts:

1. phase profiles showing where within the rotation the distances occur;
2. the average real–v2e distance compared with corresponding real–real variation; and
3. interpretation against the calibrated ruler.

Only Part 2 is prototyped below. The phase-profile and ruler figures will be designed separately.


## Part 2 — average distances across F1–F5

The reference is real period 0. Both comparison groups use the same two target periods and the same 40 phase locations per period in every recording:

- **real–real:** reference real period 0 compared with real periods 1–2;
- **real–v2e:** reference real period 0 compared with v2e periods 1–2.

The identity comparison between real period 0 and itself is excluded. Each point is the mean of 80 window comparisons; whiskers show the 5th–95th percentile across those windows. Two layouts are tested below: a colour overlay and a side-by-side recording matrix. Metrics remain in their own raw units.


In [3]:
from IPython.display import display
display = lambda *args, **kwargs: None  # suppress the superseded F1-only prototype table

F1_TEST4_RUN = (
    ROOT / "optical_chopper_data_f1/test4_synthetic/20260825_221422_w5000us"
)
f1_test4 = pd.read_csv(F1_TEST4_RUN / "results.csv")
f1_test4 = f1_test4[
    (f1_test4.polarity_channel == "all")
    & f1_test4.source.isin(["real", "v2e"])
    & f1_test4.period_index.isin([1, 2])
].copy()

f1_average = (
    f1_test4.groupby(["metric", "source"]).distance
    .agg(
        mean="mean",
        median="median",
        q05=lambda values: values.quantile(0.05),
        q95=lambda values: values.quantile(0.95),
        n="size",
    )
    .reset_index()
)

paired = (
    f1_test4.pivot(
        index=["metric", "period_index", "window_index"],
        columns="source", values="distance",
    )
    .dropna(subset=["real", "v2e"])
    .reset_index()
)
paired["v2e_minus_real"] = paired.v2e - paired.real
paired_summary = (
    paired.groupby("metric").v2e_minus_real
    .agg(
        mean_paired_difference="mean",
        median_paired_difference="median",
        fraction_v2e_greater=lambda values: (values > 0).mean(),
    )
)

mean_table = f1_average.pivot(index="metric", columns="source", values="mean")
report = mean_table.join(paired_summary)
report["real_v2e_over_real_real"] = report.v2e / report.real
report = report.reindex(METRIC_ORDER)
display(report.rename(columns={
    "real": "real–real mean",
    "v2e": "real–v2e mean",
}).round(5))

figure = make_subplots(
    rows=len(METRIC_ORDER), cols=1, vertical_spacing=0.055,
    subplot_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
)
source_order = ["real", "v2e"]
source_labels = ["real–real", "real–v2e"]
source_colours = ["#555555", "#d1352b"]

for row, metric in enumerate(METRIC_ORDER, start=1):
    block = (
        f1_average[f1_average.metric == metric]
        .set_index("source").reindex(source_order)
    )
    values = block["mean"].to_numpy()
    figure.add_trace(go.Scatter(
        x=source_labels, y=values, mode="lines+markers+text",
        showlegend=False,
        line=dict(color="#9aa0a6", width=1.5),
        marker=dict(size=10, color=source_colours),
        text=[f"{value:.4g}" for value in values],
        textposition="top center",
        error_y=dict(
            type="data",
            array=(block.q95 - block["mean"]).to_numpy(),
            arrayminus=(block["mean"] - block.q05).to_numpy(),
            symmetric=False, thickness=1, width=6, visible=True,
        ),
        customdata=np.stack([block.q05, block.q95, block.n], axis=-1),
        hovertemplate=("%{x}<br>mean %{y:.5g}"
                       "<br>5–95%: %{customdata[0]:.5g}–%{customdata[1]:.5g}"
                       "<br>n=%{customdata[2]:.0f}<extra></extra>"),
    ), row=row, col=1)
    figure.update_yaxes(title_text="distance", rangemode="tozero", row=row, col=1)

figure.update_layout(
    title=("Test 4 — F1 average real–real and real–v2e distances"
           "<br><sup>5 ms windows; periods 1–2; mean with 5th–95th percentile</sup>"),
    template="plotly_white", height=190 * len(METRIC_ORDER), width=680,
    margin=dict(l=85, r=35, t=105, b=55),
)
# F1-only prototype superseded by the combined plots below.


In [4]:
from IPython.display import display

TEST4_RUNS = {
    "F1": ROOT / "optical_chopper_data_f1/test4_synthetic/20260825_221422_w5000us",
    "F2": ROOT / "optical_chopper_data_f2/test4_synthetic/20260825_230226_w5000us",
    "F3": ROOT / "optical_chopper_data_f3/test4_synthetic/20260826_005930_w5000us",
    "F4": ROOT / "optical_chopper_data_f4/test4_synthetic/20260826_043744_w5000us",
    "F5": ROOT / "optical_chopper_data_f5/test4_synthetic/20260826_102134_w5000us",
}

test4_frames = []
for frequency, folder in TEST4_RUNS.items():
    frame = pd.read_csv(folder / "results.csv")
    frame = frame[
        (frame.polarity_channel == "all")
        & frame.source.isin(["real", "v2e"])
        & frame.period_index.isin([1, 2])
    ].copy()
    frame["frequency"] = frequency
    test4_frames.append(frame)
test4_windows = pd.concat(test4_frames, ignore_index=True)

test4_average = (
    test4_windows.groupby(["frequency", "metric", "source"]).distance
    .agg(
        mean="mean",
        median="median",
        q05=lambda values: values.quantile(0.05),
        q95=lambda values: values.quantile(0.95),
        n="size",
    )
    .reset_index()
)

test4_paired = (
    test4_windows.pivot(
        index=["frequency", "metric", "period_index", "window_index"],
        columns="source", values="distance",
    )
    .dropna(subset=["real", "v2e"])
    .reset_index()
)
test4_paired["v2e_minus_real"] = test4_paired.v2e - test4_paired.real
paired_report = (
    test4_paired.groupby(["frequency", "metric"]).v2e_minus_real
    .agg(
        mean_paired_difference="mean",
        fraction_v2e_greater=lambda values: (values > 0).mean(),
    )
)
mean_report = test4_average.pivot(
    index=["frequency", "metric"], columns="source", values="mean"
)
combined_report = mean_report.join(paired_report)
report_order = pd.MultiIndex.from_product(
    [FREQUENCIES, METRIC_ORDER], names=["frequency", "metric"]
)
combined_report = combined_report.reindex(report_order)
display(combined_report.rename(columns={
    "real": "real–real mean", "v2e": "real–v2e mean",
}).round(5))

source_order = ["real", "v2e"]
source_labels = ["real–real", "real–v2e"]

# Version A: all recordings overlaid, using the established F1–F5 colours.
overlay = make_subplots(
    rows=len(METRIC_ORDER), cols=1, vertical_spacing=0.055,
    subplot_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
)
for row, metric in enumerate(METRIC_ORDER, start=1):
    for frequency in FREQUENCIES:
        block = (
            test4_average[(test4_average.metric == metric)
                          & (test4_average.frequency == frequency)]
            .set_index("source").reindex(source_order)
        )
        overlay.add_trace(go.Scatter(
            x=source_labels, y=block["mean"], mode="lines+markers",
            name=frequency, legendgroup=frequency, showlegend=(row == 1),
            line=dict(color=FREQUENCY_COLOURS[frequency], width=1.8),
            marker=dict(size=7),
            error_y=dict(
                type="data", array=(block.q95 - block["mean"]).to_numpy(),
                arrayminus=(block["mean"] - block.q05).to_numpy(),
                symmetric=False, thickness=0.8, width=4, visible=True,
            ),
            customdata=np.stack([block.q05, block.q95, block.n], axis=-1),
            hovertemplate=(frequency + " · %{x}<br>mean %{y:.5g}"
                           + "<br>5–95%: %{customdata[0]:.5g}–%{customdata[1]:.5g}"
                           + "<br>n=%{customdata[2]:.0f}<extra></extra>"),
        ), row=row, col=1)
    overlay.update_yaxes(title_text="distance", rangemode="tozero", row=row, col=1)
overlay.update_layout(
    title=("Test 4 — average distances across F1–F5: colour overlay"
           "<br><sup>5 ms windows; periods 1–2; mean with 5th–95th percentile</sup>"),
    template="plotly_white", height=190 * len(METRIC_ORDER), width=800,
    margin=dict(l=85, r=125, t=105, b=55),
    legend=dict(orientation="v", xanchor="left", x=1.01,
                yanchor="top", y=1.0, title_text="Recording"),
)
overlay.show()

# Version B: recordings side by side. Shared y scales within each metric row.
matrix = make_subplots(
    rows=len(METRIC_ORDER), cols=len(FREQUENCIES), shared_yaxes="rows",
    horizontal_spacing=0.035, vertical_spacing=0.055,
    subplot_titles=[
        frequency if row == 0 else ""
        for row in range(len(METRIC_ORDER)) for frequency in FREQUENCIES
    ],
)
for row, metric in enumerate(METRIC_ORDER, start=1):
    for col, frequency in enumerate(FREQUENCIES, start=1):
        block = (
            test4_average[(test4_average.metric == metric)
                          & (test4_average.frequency == frequency)]
            .set_index("source").reindex(source_order)
        )
        values = block["mean"].to_numpy()
        matrix.add_trace(go.Scatter(
            x=source_labels, y=values, mode="lines+markers+text",
            showlegend=False,
            line=dict(color=FREQUENCY_COLOURS[frequency], width=1.8),
            marker=dict(
                size=8, color=FREQUENCY_COLOURS[frequency],
                symbol=["circle", "diamond"],
            ),
            text=[f"{value:.4g}" for value in values],
            textposition="top center",
            textfont=dict(size=9, color=FREQUENCY_COLOURS[frequency]),
            error_y=dict(
                type="data", array=(block.q95 - block["mean"]).to_numpy(),
                arrayminus=(block["mean"] - block.q05).to_numpy(),
                symmetric=False, thickness=0.8, width=4, visible=True,
            ),
            customdata=np.stack([block.q05, block.q95, block.n], axis=-1),
            hovertemplate=(frequency + " · %{x}<br>mean %{y:.5g}"
                           + "<br>5–95%: %{customdata[0]:.5g}–%{customdata[1]:.5g}"
                           + "<br>n=%{customdata[2]:.0f}<extra></extra>"),
        ), row=row, col=col)
        matrix.update_xaxes(
            showticklabels=(row == len(METRIC_ORDER)), tickangle=-25, row=row, col=col
        )
        matrix.update_yaxes(rangemode="tozero", row=row, col=col)
        if col == 1:
            matrix.update_yaxes(title_text=METRIC_LABELS[metric], row=row, col=col)
matrix.update_layout(
    title=("Test 4 — average distances across F1–F5: side-by-side recordings"
           "<br><sup>recording colour by column; circle = real–real; diamond = real–v2e; labels are means</sup>"),
    template="plotly_white", height=210 * len(METRIC_ORDER), width=1300,
    margin=dict(l=105, r=30, t=105, b=95),
)
matrix.show()


real–real mean  real–v2e mean  mean_paired_difference  \
frequency metric                                                             
F1        mmd_rbf03         0.01385        0.02990                 0.01606   
          mmd_rbf15         0.04425        0.06747                 0.02322   
          mmd_rbf75         0.04733        0.08967                 0.04234   
          swd               7.02214       14.69796                 7.67581   
          chamfer          10.52568       12.98448                 2.45881   
F2        mmd_rbf03         0.02080        0.02144                 0.00064   
          mmd_rbf15         0.08674        0.08203                -0.00471   
          mmd_rbf75         0.08709        0.10792                 0.02083   
          swd              11.46388       17.44861                 5.98473   
          chamfer          13.29754       11.11478                -2.18276   
F3        mmd_rbf03         0.00331        0.01239                 0.00907   
          mmd_rbf15         0.01259        0.04892                 0.03634   
          mmd_rbf75         0.01912        0.08509                 0.06597   
          swd               3.18602       14.73034                11.54431   
          chamfer           6.15328        7.94205                 1.78877   
F4        mmd_rbf03         0.00000        0.01302                 0.01302   
          mmd_rbf15         0.00186        0.06062                 0.05876   
          mmd_rbf75         0.01152        0.09668                 0.08516   
          swd               2.23332       15.64026                13.40694   
          chamfer           5.51038        8.37187                 2.86149   
F5        mmd_rbf03         0.00032        0.01388                 0.01356   
          mmd_rbf15         0.00452        0.07444                 0.06992   
          mmd_rbf75         0.00911        0.12004                 0.11093   
          swd               1.92813       18.58262                16.65449   
          chamfer           5.24974        8.52678                 3.27705   

                     fraction_v2e_greater  
frequency metric                           
F1        mmd_rbf03                1.0000  
          mmd_rbf15                1.0000  
          mmd_rbf75                1.0000  
          swd                      0.9750  
          chamfer                  1.0000  
F2        mmd_rbf03                0.6250  
          mmd_rbf15                0.2625  
          mmd_rbf75                0.8500  
          swd                      0.8125  
          chamfer                  0.0250  
F3        mmd_rbf03                1.0000  
          mmd_rbf15                1.0000  
          mmd_rbf75                1.0000  
          swd                      1.0000  
          chamfer                  1.0000  
F4        mmd_rbf03                1.0000  
          mmd_rbf15                1.0000  
          mmd_rbf75                1.0000  
          swd                      1.0000  
          chamfer                  1.0000  
F5        mmd_rbf03                1.0000  
          mmd_rbf15                1.0000  
          mmd_rbf75                1.0000  
          swd                      1.0000  
          chamfer                  1.0000

## Part 3 — v2e distance against the ruler across F1–F5

Raw distances are plotted in separate metric panels because MMD, SWD and Chamfer have unrelated native scales. Within each metric, F1–F5 are compared against the corresponding ruler maximum and floor.

A second, separate figure reports the calibrated percentage position:

`100 × (raw v2e distance − ruler floor) / (ruler maximum − ruler floor)`

The percentage is not clipped. A value above 100% means the v2e distance lies beyond the maximum reached by that metric on the measured ruler.


In [5]:
f1_placement = pd.read_csv(TEST4_RUNS["F1"] / "placement.csv")
f1_placement = f1_placement[f1_placement.polarity_channel == "all"].copy()
f1_raw = (
    test4_average[(test4_average.frequency == "F1")
                  & (test4_average.source == "v2e")]
    [["metric", "mean"]]
    .rename(columns={"mean": "raw_v2e_distance"})
)
f1_ruler = f1_raw.merge(
    f1_placement[["metric", "floor", "ceiling"]], on="metric", how="left"
)
f1_ruler = (
    f1_ruler.set_index("metric").reindex(METRIC_ORDER).reset_index()
    .rename(columns={"floor": "ruler_floor", "ceiling": "ruler_maximum"})
)
f1_ruler["ruler_percentage"] = (
    100.0
    * (f1_ruler.raw_v2e_distance - f1_ruler.ruler_floor)
    / (f1_ruler.ruler_maximum - f1_ruler.ruler_floor)
)
_f1_ruler_table = f1_ruler.round({
    "raw_v2e_distance": 5, "ruler_floor": 5,
    "ruler_maximum": 5, "ruler_percentage": 1,
})

metric_labels = [METRIC_LABELS[metric] for metric in f1_ruler.metric]
metric_colours = [METRIC_COLOURS[metric] for metric in f1_ruler.metric]
ruler_figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.17,
    subplot_titles=[
        "Raw real–v2e distance and measured ruler range",
        "Position across the calibrated ruler range",
    ],
)
ruler_figure.add_trace(go.Bar(
    x=metric_labels, y=f1_ruler.raw_v2e_distance, name="raw real–v2e distance",
    width=0.28,
    marker_color=metric_colours,
    text=[f"{value:.4g}" for value in f1_ruler.raw_v2e_distance],
    textposition="outside", cliponaxis=False,
    hovertemplate="%{x}<br>raw distance %{y:.5g}<extra></extra>",
), row=1, col=1)
ruler_figure.add_trace(go.Bar(
    x=metric_labels, y=f1_ruler.ruler_maximum, name="ruler maximum",
    width=0.28,
    marker_color="#9aa0a6",
    text=[f"{value:.4g}" for value in f1_ruler.ruler_maximum],
    textposition="outside", cliponaxis=False,
    hovertemplate="%{x}<br>ruler maximum %{y:.5g}<extra></extra>",
), row=1, col=1)
ruler_figure.add_trace(go.Scatter(
    x=metric_labels, y=f1_ruler.ruler_floor, name="ruler floor",
    mode="markers", marker=dict(color="#222222", size=8, symbol="diamond"),
    hovertemplate="%{x}<br>ruler floor %{y:.5g}<extra></extra>",
), row=1, col=1)
ruler_figure.add_trace(go.Bar(
    x=metric_labels, y=f1_ruler.ruler_percentage, name="ruler position",
    width=0.42,
    marker_color=metric_colours, showlegend=False,
    text=[f"{value:.1f}%" for value in f1_ruler.ruler_percentage],
    textposition="outside", cliponaxis=False,
    hovertemplate="%{x}<br>%{y:.1f}% of calibrated ruler range<extra></extra>",
), row=2, col=1)
ruler_figure.add_hline(
    y=100, line_dash="dash", line_color="#555555",
    annotation_text="ruler maximum", row=2, col=1,
)
ruler_figure.update_yaxes(title_text="raw distance", rangemode="tozero", row=1, col=1)
ruler_figure.update_yaxes(title_text="ruler position (%)", rangemode="tozero", row=2, col=1)
ruler_figure.update_xaxes(title_text="metric", row=2, col=1)
ruler_figure.update_layout(
    title=("Test 4 — F1 v2e distance against the temporal ruler"
           "<br><sup>raw distance uses periods 1–2; percentage is floor-corrected and not clipped</sup>"),
    barmode="group", bargap=0.42, bargroupgap=0.08,
    template="plotly_white", height=720, width=900,
    margin=dict(l=95, r=40, t=110, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
# F1-only ruler prototype superseded by the cross-frequency figures below.


In [6]:
ruler_frames = []
for frequency in FREQUENCIES:
    placement = pd.read_csv(TEST4_RUNS[frequency] / "placement.csv")
    placement = placement[placement.polarity_channel == "all"].copy()
    raw = (
        test4_average[(test4_average.frequency == frequency)
                      & (test4_average.source == "v2e")]
        [["metric", "mean"]]
        .rename(columns={"mean": "raw_v2e_distance"})
    )
    table = raw.merge(
        placement[["metric", "floor", "ceiling"]], on="metric", how="left"
    )
    table["frequency"] = frequency
    ruler_frames.append(table)
ruler_summary = pd.concat(ruler_frames, ignore_index=True).rename(columns={
    "floor": "ruler_floor", "ceiling": "ruler_maximum",
})
ruler_summary["ruler_percentage"] = (
    100.0
    * (ruler_summary.raw_v2e_distance - ruler_summary.ruler_floor)
    / (ruler_summary.ruler_maximum - ruler_summary.ruler_floor)
)
ruler_summary["metric_order"] = ruler_summary.metric.map(
    {metric: index for index, metric in enumerate(METRIC_ORDER)}
)
ruler_summary["frequency_order"] = ruler_summary.frequency.map(
    {frequency: index for index, frequency in enumerate(FREQUENCIES)}
)
ruler_summary = ruler_summary.sort_values(["metric_order", "frequency_order"])
display(ruler_summary[[
    "frequency", "metric", "raw_v2e_distance",
    "ruler_floor", "ruler_maximum", "ruler_percentage",
]].round({
    "raw_v2e_distance": 5, "ruler_floor": 5,
    "ruler_maximum": 5, "ruler_percentage": 1,
}))

# Figure 1: raw distances, one scale per metric.
raw_ruler_figure = make_subplots(
    rows=len(METRIC_ORDER), cols=1, shared_xaxes=True, vertical_spacing=0.055,
    subplot_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
)
frequency_colours = [FREQUENCY_COLOURS[frequency] for frequency in FREQUENCIES]
for row, metric in enumerate(METRIC_ORDER, start=1):
    block = (
        ruler_summary[ruler_summary.metric == metric]
        .set_index("frequency").reindex(FREQUENCIES)
    )
    raw_ruler_figure.add_trace(go.Bar(
        x=FREQUENCIES, y=block.raw_v2e_distance,
        name="raw real–v2e distance", legendgroup="raw", showlegend=(row == 1),
        width=0.28, marker_color=frequency_colours,
        text=[f"{value:.4g}" for value in block.raw_v2e_distance],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>raw distance %{y:.5g}<extra></extra>",
    ), row=row, col=1)
    raw_ruler_figure.add_trace(go.Bar(
        x=FREQUENCIES, y=block.ruler_maximum,
        name="ruler maximum", legendgroup="maximum", showlegend=(row == 1),
        width=0.28, marker_color="#b7b9bd",
        text=[f"{value:.4g}" for value in block.ruler_maximum],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>ruler maximum %{y:.5g}<extra></extra>",
    ), row=row, col=1)
    raw_ruler_figure.add_trace(go.Scatter(
        x=FREQUENCIES, y=block.ruler_floor,
        name="ruler floor", legendgroup="floor", showlegend=(row == 1),
        mode="markers", marker=dict(color="#222222", size=7, symbol="diamond"),
        hovertemplate="%{x}<br>ruler floor %{y:.5g}<extra></extra>",
    ), row=row, col=1)
    raw_ruler_figure.update_yaxes(title_text="distance", rangemode="tozero", row=row, col=1)
raw_ruler_figure.update_xaxes(title_text="recording", row=len(METRIC_ORDER), col=1)
raw_ruler_figure.update_layout(
    title=("Test 4 — raw real–v2e distance against each metric's ruler"
           "<br><sup>F1–F5 within each metric; raw distance uses periods 1–2</sup>"),
    barmode="group", bargap=0.42, bargroupgap=0.08,
    template="plotly_white", height=205 * len(METRIC_ORDER), width=820,
    margin=dict(l=90, r=40, t=110, b=65),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
raw_ruler_figure.show()

# Figure 2: floor-corrected percentages on a common scale.
percentage_figure = go.Figure()
metric_labels = [METRIC_LABELS[metric] for metric in METRIC_ORDER]
for frequency in FREQUENCIES:
    block = (
        ruler_summary[ruler_summary.frequency == frequency]
        .set_index("metric").reindex(METRIC_ORDER)
    )
    percentage_figure.add_trace(go.Bar(
        x=metric_labels, y=block.ruler_percentage, name=frequency,
        marker_color=FREQUENCY_COLOURS[frequency], width=0.12,
        text=[f"{value:.1f}%" for value in block.ruler_percentage],
        textposition="outside", textfont=dict(size=9), cliponaxis=False,
        hovertemplate=(frequency + " · %{x}<br>%{y:.1f}% of calibrated ruler range"
                       + "<extra></extra>"),
    ))
percentage_figure.add_hline(
    y=100, line_dash="dash", line_color="#555555",
    annotation_text="ruler maximum",
)
percentage_figure.update_layout(
    title=("Test 4 — v2e position across each calibrated ruler range"
           "<br><sup>0% = ruler floor; 100% = ruler maximum; values are not clipped</sup>"),
    xaxis_title="metric", yaxis_title="ruler position (%)",
    barmode="group", bargap=0.28, bargroupgap=0.05,
    template="plotly_white", height=560, width=1000,
    margin=dict(l=90, r=40, t=110, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0,
                title_text="Recording"),
)
percentage_figure.show()


,frequency,metric,raw_v2e_distance,ruler_floor,ruler_maximum,ruler_percentage
1,F1,mmd_rbf03,0.02990,0.00026,0.02321,129.2
6,F2,mmd_rbf03,0.02144,0.00036,0.02374,90.2
11,F3,mmd_rbf03,0.01239,0.00016,0.02307,53.4
16,F4,mmd_rbf03,0.01302,0.00015,0.02275,56.9
21,F5,mmd_rbf03,0.01388,0.00020,0.02257,61.1
2,F1,mmd_rbf15,0.06747,0.00023,0.15365,43.8
7,F2,mmd_rbf15,0.08203,0.00033,0.15674,52.2
12,F3,mmd_rbf15,0.04892,0.00044,0.15540,31.3
17,F4,mmd_rbf15,0.06062,0.00029,0.15466,39.1
22,F5,mmd_rbf15,0.07444,0.00025,0.15433,48.1


## Test 4 — real, v2e and V2CE across F1–F5

This view uses the completed three-source runs. It compares periods 1–2 against the real baseline period, excluding period 0 self-comparisons so the real–real reference reflects variation between rotations. Bars show the mean and whiskers show the 5th–95th percentile.


In [7]:
# Re-load the completed three-source runs so this cell can be rerun after updates.
V2CE_TEST4_INDEX = ROOT.parent / "v2ce/test4_run_index.yaml"
ANALYSIS_PERIODS = [1, 2]  # Exclude baseline period 0 self-comparisons.
SOURCE_ORDER = ["real", "v2e", "v2ce"]
SOURCE_LABELS = {
    "real": "real–real",
    "v2e": "real–v2e",
    "v2ce": "real–V2CE",
}
SOURCE_COLOURS = {
    "real": "#62666d",
    "v2e": "#c44e52",
    "v2ce": "#4c78a8",
}

run_index = yaml.safe_load(V2CE_TEST4_INDEX.read_text(encoding="utf-8"))
three_source_frames = []
for frequency_key, run_folder in run_index.items():
    frame = pd.read_csv(Path(run_folder) / "results.csv")
    frame = frame[
        (frame.polarity_channel == "all")
        & frame.source.isin(SOURCE_ORDER)
        & frame.period_index.isin(ANALYSIS_PERIODS)
    ].dropna(subset=["distance"]).copy()
    frame["frequency"] = frequency_key.upper()
    three_source_frames.append(frame)

three_source_windows = pd.concat(three_source_frames, ignore_index=True)
three_source_average = (
    three_source_windows
    .groupby(["frequency", "metric", "source"], as_index=False)
    .distance.agg(
        mean="mean",
        q05=lambda values: values.quantile(0.05),
        q95=lambda values: values.quantile(0.95),
        comparisons="count",
    )
)

three_source_figure = make_subplots(
    rows=3, cols=2,
    subplot_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
    horizontal_spacing=0.11, vertical_spacing=0.14,
)

for metric_index, metric in enumerate(METRIC_ORDER):
    row, col = metric_index // 2 + 1, metric_index % 2 + 1
    metric_block = three_source_average[three_source_average.metric == metric]

    for source in SOURCE_ORDER:
        block = (
            metric_block[metric_block.source == source]
            .set_index("frequency").reindex(FREQUENCIES)
        )
        values = block["mean"].to_numpy()
        digits = 3 if metric.startswith("mmd_") else 2
        three_source_figure.add_trace(go.Bar(
            x=FREQUENCIES, y=values,
            name=SOURCE_LABELS[source],
            legendgroup=source, showlegend=(metric_index == 0),
            marker_color=SOURCE_COLOURS[source], width=0.19,
            text=[f"{value:.{digits}f}" for value in values],
            textposition="outside", textfont=dict(size=10), cliponaxis=False,
            error_y=dict(
                type="data",
                array=(block.q95 - block["mean"]).to_numpy(),
                arrayminus=(block["mean"] - block.q05).to_numpy(),
                thickness=1, width=3,
            ),
            customdata=np.column_stack([block.q05, block.q95, block.comparisons]),
            hovertemplate=(
                SOURCE_LABELS[source] + "<br>%{x}<br>mean=%{y:.5g}"
                + "<br>5th–95th=%{customdata[0]:.5g}–%{customdata[1]:.5g}"
                + "<br>comparisons=%{customdata[2]:.0f}<extra></extra>"
            ),
        ), row=row, col=col)

    panel_maximum = metric_block.q95.max()
    three_source_figure.update_yaxes(
        title_text="mean distance", range=[0, panel_maximum * 1.24],
        row=row, col=col,
    )
    three_source_figure.update_xaxes(title_text="recording", row=row, col=col)

three_source_figure.update_layout(
    title=("Test 4 — real, v2e and V2CE across F1–F5"
           "<br><sup>5 ms windows; periods 1–2; bars are means; whiskers are 5th–95th percentiles</sup>"),
    barmode="group", bargap=0.30, bargroupgap=0.07,
    template="plotly_white", height=1080, width=1120,
    margin=dict(l=75, r=35, t=120, b=65),
    legend=dict(orientation="h", yanchor="bottom", y=1.035, xanchor="center", x=0.5),
)
three_source_figure.show()


## Qualitative check — one matched phase window in 3D

The three panels use identical axes. Real is the baseline-period window used by Test 4; v2e and V2CE are taken from the selected comparison period at the same phase. Colour repeats the relative timestamp shown on the vertical axis. Subplot titles report the raw number of events and the number rendered.


In [8]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

# Change these four values to inspect another comparison already in Test 4.
PROJECT_ROOT = Path("C:/Users/cxm3593/Academic/Workspace/EventSimilarityAnalysis")
FREQUENCY = "F1"       # F1, F2, F3, F4 or F5
PERIOD_INDEX = 1         # The notebook analysis uses periods 1 and 2.
PHASE_POSITION = 20      # Position 0–39 within the 40 selected phase windows.
MAX_PLOT_EVENTS = 20_000 # Rendering cap only; raw counts remain visible.

run_index_path = PROJECT_ROOT / "output/v2ce/test4_run_index.yaml"
run_index = yaml.safe_load(run_index_path.read_text(encoding="utf-8"))
run_folder = Path(run_index[FREQUENCY.lower()])
run_config = yaml.safe_load((run_folder / "run_config.yaml").read_text(encoding="utf-8"))
results = pd.read_csv(run_folder / "results.csv", usecols=[
    "source", "period_index", "metric", "window_index",
    "window_start_us", "polarity_channel",
])

available_windows = (
    results[
        (results.source == "real")
        & (results.period_index == PERIOD_INDEX)
        & (results.metric == "mmd_rbf15")
        & (results.polarity_channel == "all")
    ]
    .drop_duplicates("window_index")
    .sort_values("window_index")
    .reset_index(drop=True)
)
if not 0 <= PHASE_POSITION < len(available_windows):
    raise ValueError(f"PHASE_POSITION must be between 0 and {len(available_windows) - 1}")
selected = available_windows.iloc[PHASE_POSITION]

parameters = run_config["parameters"]
window_us = int(parameters["window_us"])
rotation_period_us = int(parameters["rotation_period_us"])
baseline_period = int(parameters["baseline_period"])
window_index = int(selected.window_index)
phase_offset_us = window_index * window_us
target_start_us = int(selected.window_start_us)

real_path = Path(run_config["trial"]["real_path"])
v2e_path = Path(run_config["trial"]["v2e_path"])
v2ce_path = Path(parameters["simulator_paths"]["v2ce"])

def h5_lower_bound(dataset, timestamp_us):
    """Binary search a sorted HDF5 event dataset without loading every timestamp."""
    low, high = 0, len(dataset)
    while low < high:
        middle = (low + high) // 2
        if int(dataset[middle]["t"]) < timestamp_us:
            low = middle + 1
        else:
            high = middle
    return low

def first_timestamp(path):
    with h5py.File(path, "r") as handle:
        return int(handle["events"][0]["t"])

def read_event_window(path, start_us, width_us):
    with h5py.File(path, "r") as handle:
        events = handle["events"]
        low = h5_lower_bound(events, start_us)
        high = h5_lower_bound(events, start_us + width_us)
        return events[low:high]

def rendering_sample(events, maximum, seed):
    if maximum is None or len(events) <= maximum:
        return events
    rng = np.random.default_rng(seed)
    chosen = np.sort(rng.choice(len(events), size=maximum, replace=False))
    return events[chosen]

real_start_us = (
    first_timestamp(real_path)
    + baseline_period * rotation_period_us
    + phase_offset_us
)
cloud_specs = [
    ("real", real_path, real_start_us),
    ("v2e", v2e_path, target_start_us),
    ("V2CE", v2ce_path, target_start_us),
]
clouds = []
for seed, (label, path, start_us) in enumerate(cloud_specs):
    raw = read_event_window(path, start_us, window_us)
    plotted = rendering_sample(raw, MAX_PLOT_EVENTS, seed)
    clouds.append({
        "label": label, "start_us": start_us,
        "raw": raw, "plotted": plotted,
    })

subplot_titles = [
    f"{cloud['label']} — {len(cloud['raw']):,} events"
    + (f" ({len(cloud['plotted']):,} shown)" if len(cloud['plotted']) < len(cloud['raw']) else "")
    for cloud in clouds
]
phase_cloud_figure = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=subplot_titles, horizontal_spacing=0.015,
)

for column, cloud in enumerate(clouds, start=1):
    events = cloud["plotted"]
    relative_time_ms = (events["t"].astype(np.float64) - cloud["start_us"]) / 1000.0
    phase_cloud_figure.add_trace(go.Scatter3d(
        x=events["x"], y=events["y"], z=relative_time_ms,
        mode="markers", name=cloud["label"], showlegend=False,
        marker=dict(
            size=1.6, opacity=0.48, color=relative_time_ms,
            colorscale="Viridis", cmin=0, cmax=window_us / 1000.0,
            showscale=(column == 3),
            colorbar=dict(title="time (ms)", len=0.62, x=1.01),
        ),
        customdata=events["p"],
        hovertemplate=(
            cloud["label"]
            + "<br>x=%{x} px<br>y=%{y} px<br>t=%{z:.3f} ms"
            + "<br>polarity=%{customdata}<extra></extra>"
        ),
    ), row=1, col=column)

scene_settings = dict(
    xaxis=dict(title="x (px)", range=[0, 1280]),
    yaxis=dict(title="y (px)", range=[720, 0]),
    zaxis=dict(title="time in window (ms)", range=[0, window_us / 1000.0]),
    aspectmode="manual", aspectratio=dict(x=1.55, y=0.88, z=0.72),
    camera=dict(eye=dict(x=1.45, y=-1.45, z=1.0)),
)
for scene_name in ("scene", "scene2", "scene3"):
    phase_cloud_figure.layout[scene_name].update(scene_settings)

phase_cloud_figure.update_layout(
    title=(
        f"Test 4 qualitative phase window — {FREQUENCY}, phase position {PHASE_POSITION}/39"
        f"<br><sup>window index {window_index}; offset {phase_offset_us / 1000:.1f} ms; "
        f"real baseline period {baseline_period} versus simulator period {PERIOD_INDEX}; "
        f"window {window_us / 1000:.1f} ms</sup>"
    ),
    template="plotly_white", height=590, width=1250,
    margin=dict(l=0, r=75, t=115, b=0),
)
phase_cloud_figure.show()
